# <font color="blue"><b>From Instruction Fine-Tuning to QLoRA: A Cost–Quality Study on Medical Instruction Tuning</b></font>

---

## Problem Statement

Large language models are expensive to adapt. **Full fine-tuning** updates every weight in the model — it works well but needs a lot of GPU memory and compute, which puts it out of reach on modest hardware. Two parameter-efficient methods promise most of the quality for a fraction of the cost:

- **LoRA** freezes the model and trains only small low-rank adapter matrices (well under 1% of the parameters).
- **QLoRA** goes further — it stores the frozen model in **4-bit**, then trains the same LoRA adapters on top, cutting memory again.

In this assignment you will put these three methods on a real task: teaching **Qwen2.5-0.5B-Instruct** to answer medical instructions from the **AlpaCare-MedInstruct-52k** dataset (each example is `instruction (+ optional input) → reference answer`). You will fine-tune the **same model on the same data three ways**, then measure what each approach costs and what it buys.

**The central question:** *given a fixed model, a fixed dataset, and a single free T4 GPU, is it worth updating all ~500M parameters — or does a low-rank slice (LoRA), or a low-rank slice over a 4-bit backbone (QLoRA), get you almost the same answer quality for far less memory?* You will also compare **two LoRA ranks (r = 8 and r = 16)** for both LoRA and QLoRA, to see how adapter capacity trades against cost.

---

## What You Will Do

You will build the pipeline end-to-end, as a hand-written training loop (a torch `Dataset`, a `collate` function, and `DataLoader`s — no `Trainer`), reused across all three methods:

1. **Imports & seed** — set up the environment reproducibly. *(provided)*
2. **Configuration** — model, data sizes, and all hyperparameters in one place. *(provided)*
3. **Helper functions** — LoRA/QLoRA configs, parameter counting, memory utilities. *(provided)*
4. **Load dataset + statistics** — take 300 / 30 / 30 for train / val / test; inspect fields and lengths.
5. **Dataset, collate & loaders** — build the prompt with the chat template, mark answer tokens with an offset-based mask, pad batches, and create the three loaders.
6. **Model load + IFT training** — write the masked-loss + shared training loop, then run **full instruction fine-tuning**.
7. **LoRA** — freeze the backbone and train adapters at **r = 8 and r = 16**.
8. **QLoRA** — repeat the rank sweep with a **4-bit NF4** backbone.
9. **Test evaluation** — greedy-generate on the test set and score every run with **ROUGE-L** and **BERTScore-F1**.
10. **Plots** — per-epoch train vs validation loss for every run.
11. **Error analysis** — inspect the best model's answers vs the references and summarise the failure modes.
12. **Parameter analysis** — compare quality against % trainable parameters, peak memory, and training time.

**How the notebook is organised**
- Cells marked **`# ── PROVIDED ──`** are already written — just run them in order.
- You only need to write the code inside **`## CODE HERE ##`**.
- Every step with a red **`[marks]`** tag is graded: complete the code where indicated and write the **`Your Answer`** reflection.
- Reflections must **cite the actual numbers your run produced** — not generic descriptions.

**Runtime:** Colab **free T4 GPU**. We deliberately use a small data subset and short runs so all methods finish in one session. Select the T4 runtime and run top-to-bottom after a restart. No API key needed.

> **Safety note.** This is an exercise on a 0.5B model with machine-generated references. Its outputs are **not** clinically reliable and must never guide real medical decisions.

---

## Mark Distribution

| Step | Topic | Marks |
|------|-------|-------|
| 1 | Imports & seed | 0 *(provided)* |
| 2 | Configuration | 0 *(provided)* |
| 3 | Helper functions (LoRA / QLoRA) | 0 *(provided)* |
| 4 | Load dataset + statistics | 8 |
| 5 | Dataset, collate & loaders | 14 |
| 6 | Model load + IFT training | 20 |
| 7 | LoRA (r = 8 and r = 16) | 14 |
| 8 | QLoRA (r = 8 and r = 16) | 14 |
| 9 | Test evaluation (ROUGE-L + BERTScore) | 10 |
| 10 | Plots (train vs validation) | 6 |
| 11 | Error analysis | 8 |
| 12 | Parameter analysis | 6 |
| | **Total** | **100** |

Most marks sit where the core learning is: the training loop and full IFT (Step 6), and the data pipeline plus the two PEFT methods (Steps 5, 7, 8). Steps 1–3 are provided scaffolding and carry no marks.

---

In [ ]:
# Some package installation.
!pip install -q evaluate rouge_score bert_score "bitsandbytes>=0.43"

In [ ]:
# Avoid a torchao/T4 incompatibility that can surface with peft/quantised stacks.
!pip uninstall -y -q torchao

### <font color="blue"><b>Step 1: Imports and Seed Setting</b></font>  *(provided — just run)*

Import the libraries used throughout — PyTorch and `torch.utils.data`, Hugging Face `transformers` / `datasets` / `peft`, and plotting — and fix a single random seed across `random`, NumPy, and PyTorch (`seed_everything`) so every run is reproducible.

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
# 1. Imports and Seed Setting
# ---PROVIDED ------------------------------------------------
# ============================================================
import os, gc, math, time, random, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, get_cosine_schedule_with_warmup,
                          set_seed)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training,
                  TaskType)

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

seed_everything()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### <font color="blue"><b>Step 2: Configuration</b></font>  *(provided — just run)*

All the knobs in one place: the model ID, dataset split sizes (`N_TRAIN` / `N_VAL` / `N_TEST`), the sequence-length budget (`MAX_LEN`), and the training hyperparameters — batch size, gradient accumulation, epochs, the two learning rates (`LR_FULL`, `LR_LORA`), and the LoRA ranks. The rest of the notebook reads from these constants, so change behaviour here rather than editing later cells.

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
#  Step 2: Configuration
# ---PROVIDED ------------------------------------------------
# ============================================================
MODEL_ID   = "Qwen/Qwen2.5-0.5B-Instruct"

# ---- Data ----
N_TRAIN    = 300
N_VAL      = 30
N_TEST     = 30
MAX_LEN    = 256        # token budget per (prompt + answer) example

# ---- Training ----
BATCH_SIZE = 4          # small: fits a free T4 - if not fit please reduce it
GRAD_ACCUM = 1          # effective batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS     = 3
WARMUP_RATIO  = 0.05
MAX_GRAD_NORM = 1.0

LR_FULL    = 1e-5       # full fine-tuning: small LR (all weights move)
LR_LORA    = 2e-4       # LoRA / QLoRA: frozen backbone -> larger LR is safe

# ---- LoRA / QLoRA ----
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ---- Evaluation ----
EVAL_N     = 60         # number of test prompts to generate on when scoring

### <font color="blue"><b>Step 3: Helper Functions</b></font>  *(provided — just run)*

Small utilities reused across every method: `count_trainable` (trainable-vs-total parameter counts), `get_lora_config` / `get_bnb_config` (the shared LoRA and 4-bit NF4 quantisation settings), and `gpu_mem_gb` / `free_memory` (peak-memory reporting and VRAM cleanup between runs).

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
#  Step 3: Helper Functions (LoRA / QLoRA)
# ---PROVIDED ------------------------------------------------
# ============================================================

def count_trainable(model):
    """Return (trainable, total, percent-trainable) parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total, 100 * trainable / total


def get_lora_config():
    """Standard LoRA config for Qwen2.5 (attention + MLP projections)."""
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )


def get_bnb_config():
    """4-bit NF4 quantisation config for QLoRA (fp16 compute for T4)."""
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )


def gpu_mem_gb():
    """Peak GPU memory used since the last reset, in GB."""
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


def free_memory(*objs):
    """Delete big objects (models/trainers) and reclaim GPU memory between runs."""
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

## <font color="blue"><b>Step 4: Load Dataset and Statistics</b></font>  <font color="red">**[8 marks]**</font>

Load **AlpaCare-MedInstruct-52k** and randomly take **300 train / 30 validation / 30 test**. Each row is just `instruction (+ optional input) → output`, where `output` is a reference answer. The dataset stores the literal token `<noinput>` when there is no extra context, so blank that out first.

Then compute a few basic statistics on the training split: the fraction of rows with a non-empty `input`, and the token-length distribution of `instruction + input + output` (mean / median / p95 / max), plus a length histogram.

**Marks:** code **[<font color="red">4 marks</font>]** — correct load, `<noinput>` handling, 300/30/30 split, and the statistics + histogram; reflection **[<font color="red">4 marks</font>]** — see below.

In [ ]:
# ============================================================
#  Step 4: Load Dataset and Statistics
# ============================================================

# load the dataset and convert to pandas DataFrame
## CODE HERE ##

# The dataset stores the literal token "<noinput>" when there is no extra context.
## CODE HERE ##

# Shuffle once, then slice off the splits by index.
## CODE HERE ##

# ---- Basic statistics on the training split ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

frac_with_input = (train_df["input"] != "").mean()

def combined_len(r):
    text = r["instruction"] + " " + r["input"] + " " + r["output"]
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

## CODE HERE ##

### ✍️ Step 4 Reflection  <font color="red">[4 marks]</font>

**[Your Answer]** must cover:
- The three fields (`instruction`, `input`, `output`), what each holds, and roughly what fraction of rows use a non-empty `input`. **<font color="red">1 marks</font>**
- The token-length statistics you printed (mean / median / p95 / max) and the `MAX_LEN` you will adopt because of them — justify the choice against the p95. **<font color="red">2 marks</font>**
- That `output` is a *machine-generated* reference (text-davinci-003), and what that implies for treating it as ground truth when scoring models later. **<font color="red">1 marks</font>**

*Cite the actual numbers your run produced.*

## <font color="blue"><b>Step 5: Dataset, Collate, and DataLoaders</b></font>  <font color="red">**[14 marks]**</font>

Build the input pipeline as a hand-written `torch.utils.data.Dataset` (like the DPO demo), with tokenisation deferred to a `collate` function:

- **`SFTDataset`** — holds `(prompt, answer)` text pairs. The prompt is rendered with the tokenizer's **chat template** (instruction → *user* turn, primed to answer).
- **`tokenize_example`** — tokenises the full `prompt + answer` **once** and uses the **offset mapping** to mark exactly where the answer begins, producing a `comp_mask` (1 = answer token, 0 = prompt token).
- **`collate`** — pads a batch to its longest sequence and returns `input_ids`, `attn`, and `comp_mask`.
- **Three DataLoaders** — `train_loader` (shuffled), `val_loader`, `test_loader`.

Finally, print the **first full training batch** to confirm the shapes and the answer/prompt masking.

**Marks:** code <font color="red">**[10 marks]**</font> — `SFTDataset` + chat-template prompt, offset-based `comp_mask`, correct padded `collate`, three loaders, and the batch print; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 5: Dataset, Collate, and DataLoaders
# ============================================================

SYSTEM_MSG = "You are a helpful medical assistant. Answer accurately and safely."

def build_prompt(instruction, input_text):
    ## CODE HERE ##
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


class SFTDataset(Dataset):
    """Holds (prompt_text, answer_text) pairs; tokenisation is deferred to collate()."""
    def __init__(self, frame):
        self.rows = [{"prompt": build_prompt(r["instruction"], r["input"]),
                      "answer": r["output"].strip()}
                     for _, r in frame.iterrows()]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return self.rows[i]


def tokenize_example(prompt, answer):
    """
    Tokenise the full (prompt + answer) once and use the offset mapping to mark
    exactly where the answer begins. comp_mask = 1 on answer tokens, 0 on prompt.
    Tokenising once avoids token-merge drift at the boundary.
    """
    ## CODE HERE ##
    return ids, mask


def collate(batch):
    """Pad a batch to its longest sequence. Returns input_ids, attn, comp_mask."""
    ## CODE HERE ##
    return input_ids, attn, comp_mask


# ---- Three DataLoaders ----
## CODE HERE ##

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# ---- Print the first FULL batch of the train loader ----
## CODE HERE ##*

### ✍️ Step 5 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- Why the prompt uses the tokenizer's **chat template** for an `-Instruct` model (special tokens must match what the model saw in pre-training). <font color="red">**[1]**</font>
- What `comp_mask` encodes and why we compute the loss on the **answer tokens only** (completion-only). <font color="red">**[1]**</font>
- Why tokenising `prompt + answer` **once with offsets** is more correct than tokenising them separately and concatenating (avoids token-merge drift at the boundary). <font color="red">**[1]**</font>
- From the printed batch: the supervised-vs-masked token counts, and why padding fills `input_ids` with the pad token but `comp_mask` with 0. <font color="red">**[1]**</font>

*Cite the actual numbers your run produced.*

## <font color="blue"><b>Step 6: Model Loading and Full Instruction Fine-Tuning (IFT)</b></font>  <font color="red">**[20 marks]**</font>

This is the core of the notebook. First define the training machinery that all three methods reuse, then run full fine-tuning:

- **`batch_loss`** — the causal-LM next-token loss, but computed over **answer tokens only** (using `comp_mask`), so the model is never scored on re-typing the prompt.
- **`eval_loss`** — averages `batch_loss` over a whole loader (no gradients), used for validation.
- **`train_model`** — the shared manual training loop: gradient accumulation, warmup → cosine schedule, gradient clipping, per-epoch train/val history, and early stopping on validation loss. Returns the best model plus its history, training time, and peak memory.

Then load Qwen2.5-0.5B in **fp32** (pure fp16 full fine-tuning overflows on this model), confirm ~100% of parameters are trainable, run **full IFT** with the small learning rate `LR_FULL`, and print the per-epoch train/val losses.

**Marks:** code <font color="red">**[14 marks]**</font> — correct answer-only `batch_loss`, the `train_model` loop (accumulation, schedule, clipping, early stopping, history), fp32 model load, and the IFT run with per-epoch report; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 6: Load Model and Full Instruction Fine-Tuning (IFT)
# ============================================================

# ---- Loss over ANSWER tokens only (comp_mask == 1) ----
def batch_loss(model, input_ids, attn, comp_mask):
    """Mean next-token NLL over answer tokens for ONE batch."""
    ## CODE HERE ##
    return -(tok * mask).sum() / mask.sum().clamp(min=1.0)


@torch.no_grad()
def eval_loss(model, loader):
    """Average batch_loss over an entire loader (no gradients)."""
    ## CODE HERE ##
    return tot / max(n, 1)


# ---- Shared training loop (reused for IFT, LoRA, QLoRA) ----
def train_model(model, lr, tag):
    """
    Manual training loop: gradient accumulation, warmup->cosine schedule,
    gradient clipping, and early stopping on validation loss.
    Returns (best_model, hist, train_time_s, peak_mem_gb).
    hist keys: 'train_loss' & 'val_loss' (per epoch), 'step_loss' (per step).
    """
    seed_everything()
    free_memory()

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    trainable_keys = {n for n, p in model.named_parameters() if p.requires_grad}   # only these change
    total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(WARMUP_RATIO * total_steps), total_steps)

    hist = {"train_loss": [], "val_loss": [], "step_loss": []}
    best_val, best_epoch, best_state, no_improve, PATIENCE = float("inf"), 0, None, 0, 2

    t0 = time.time()

    ## CODE HERE ##

    train_time, peak_mem = time.time() - t0, gpu_mem_gb()
    print(f"[{tag}] best epoch = {best_epoch} | time = {train_time:.1f}s | peak mem = {peak_mem:.2f} GB")
    return model, hist, train_time, peak_mem


# ---- Load model for Full IFT ----
# fp32 (not fp16): pure-fp16 full fine-tuning overflows on Qwen2.5 and gives a
# garbage loss in the hundreds. fp32 is fine for a 0.5B model on a T4.
seed_everything()
free_memory()

## CODE HERE ##

# ---- Per-epoch loss history ----
print("\nEpoch |  Train Loss | Val Loss")
print("-" * 34)
for ep, (tl, vl) in enumerate(zip(full_hist["train_loss"], full_hist["val_loss"]), 1):
    print(f"  {ep:2d}  |   {tl:.4f}   | {vl:.4f}")

### ✍️ Step 6 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- How `batch_loss` supervises **only** the answer tokens (the `comp_mask` shift + masked mean), and how completion-only loss differs from training on the whole sequence. <font color="red">**[2 marks]**</font>
- Where warmup, cosine decay, gradient clipping, and gradient accumulation appear in `train_model`, and your **effective batch size** (`BATCH_SIZE × GRAD_ACCUM`). <font color="red">**[2 marks]**</font>
- Why full IFT is loaded in **fp32** (pure fp16 full fine-tuning overflows on Qwen2.5) and why its learning rate `LR_FULL = 1e-5` is deliberately small. <font color="red">**[1 marks]**</font>
- The per-epoch train/val trend — is the model improving or starting to overfit? <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

## <font color="blue"><b>Step 7: LoRA Fine-Tuning — r = 8 and r = 16</b></font>  <font color="red">**[14 marks]**</font>

Now freeze the backbone and train only small low-rank adapter matrices. For a frozen weight `W`, LoRA learns `A` and `B` and adds `(α/r)·B·A` as a correction — so only a tiny fraction of the parameters are trained.

Run LoRA at **two ranks, r = 8 and r = 16**, holding the `alpha/rank` ratio fixed (`alpha = 2·r`) so you isolate the effect of *rank* alone. Reuse the same `train_model` loop, store each run in `lora_runs[rank]`, and report the trainable-parameter percentage and per-epoch losses for both.

*(The `torchao` uninstall here avoids a T4 incompatibility that can otherwise break the peft/quantised path in the next step.)*

**Marks:** code <font color="red">**[8 marks]**</font> — correct `LoraConfig` (rank, alpha, dropout, target modules), the two-rank sweep reusing `train_model`, and the trainable-% + per-epoch report for each rank; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
LORA_RANKS = [8, 16]   # compare two LoRA ranks
# ============================================================
#  Step 7: LoRA Fine-Tuning (rank sweep: r = 8 and r = 16)
# ============================================================

def build_lora_model(rank):
    """Fresh fp32 backbone + LoRA adapters at the given rank."""
    seed_everything()
    free_memory()
    ## CODE HERE ##
    return get_peft_model(base, cfg)


# ---- Run LoRA at each rank ----
lora_runs = {}   # rank -> dict(model, hist, time, mem, pct)

for rank in LORA_RANKS:
    ## CODE HERE ##

    print("\nEpoch |  Train Loss | Val Loss")
    print("-" * 34)
    for ep, (tl, vl) in enumerate(zip(hist["train_loss"], hist["val_loss"]), 1):
        print(f"  {ep:2d}  |   {tl:.4f}   | {vl:.4f}")
    print()

    free_memory()   # clear before the next rank

### ✍️ Step 7 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- The role of the **rank `r`** and **alpha `α`** (the `α/r` scaling), and why `B` is zero-initialised (the adapter starts as identity, so training is stable from step 0). <font color="red">**[2 marks]**</font>
- The **% trainable parameters** at r = 8 vs r = 16 (from `print_trainable_parameters`), and how both compare to full IFT's ~100%. <font color="red">**[2 marks]**</font>
- The effect of doubling the rank on validation loss — did r = 16 actually help, or just cost more parameters for the same quality? Cite both runs. <font color="red">**[1 marks]**</font>
- Why LoRA can safely use a much larger learning rate (`LR_LORA = 2e-4`) than full IFT. <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

## <font color="blue"><b>Step 8: QLoRA Fine-Tuning — r = 8 and r = 16</b></font>  <font color="red">**[14 marks]**</font>

LoRA froze the backbone but still kept it in full precision, which dominates GPU memory. **QLoRA** stores that frozen backbone in **4-bit NF4** instead, then trains the *same* LoRA adapters on top — the only thing that changes from Step 7 is the backbone's precision, making this a clean controlled comparison.

Set the 4-bit recipe with `BitsAndBytesConfig` (NF4, double quantisation, fp16 compute for the T4), call `prepare_model_for_kbit_training` before attaching the adapters, and run the **same two-rank sweep (r = 8 and r = 16)**. Store each run in `qlora_runs[rank]` and report the trainable-parameter percentage and per-epoch losses.

**Marks:** code <font color="red">**[8 marks]**</font> — correct `BitsAndBytesConfig`, `prepare_model_for_kbit_training`, the same `LoraConfig` as Step 7, the two-rank sweep reusing `train_model`, and the trainable-% + per-epoch report; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 8: QLoRA Fine-Tuning (4-bit NF4 backbone; rank sweep: r = 8 and r = 16)
# ============================================================

def build_qlora_model(rank):
    """Fresh 4-bit NF4 backbone + LoRA adapters at the given rank."""
    seed_everything()
    free_memory()

    ## CODE HERE ##
    return get_peft_model(base, cfg)


# ---- Run QLoRA at each rank ----
qlora_runs = {}   # rank -> dict(model, hist, time, mem, pct)

for rank in LORA_RANKS:

    ## CODE HERE ##

    print("\nEpoch |  Train Loss | Val Loss")
    print("-" * 34)
    for ep, (tl, vl) in enumerate(zip(hist["train_loss"], hist["val_loss"]), 1):
        print(f"  {ep:2d}  |   {tl:.4f}   | {vl:.4f}")
    print()

    free_memory()   # clear before the next rank

### ✍️ Step 8 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- Each `BitsAndBytesConfig` setting in plain language — `load_in_4bit`, `nf4`, double quantisation, and `compute_dtype=float16` — and why compute is fp16 on a T4. <font color="red">**[2 marks]**</font>
- What `prepare_model_for_kbit_training` does and why it is required before attaching LoRA. <font color="red">**[1 marks]**</font>
- QLoRA vs LoRA at the **same rank**: how much peak memory did 4-bit save, and did it cost any measurable answer quality? Cite both. <font color="red">**[2 marks]**</font>
- Why QLoRA's trainable-parameter count matches LoRA's at the same rank (only the frozen backbone's precision changed). <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

## <font color="blue"><b>Step 9: Test Evaluation — IFT, LoRA, QLoRA</b></font>  <font color="red">**[10 marks]**</font>

Now measure answer quality on the held-out test set. Greedy-generate an answer for a fixed slice of test prompts (`EVAL_N`, the same prompts for every model so scores are comparable), then score the generations against the references with two complementary metrics:

- **ROUGE-L** — overlap of word sequences (did we *say* similar things?).
- **BERTScore-F1** — embedding similarity (did we *mean* similar things, even with different wording?).

Score every run — full IFT, both LoRA ranks, and both QLoRA ranks — and collect the numbers (plus each run's % trainable, peak memory, and training time) into `results`, with the generated text kept in `preds_store` for the error analysis.

**Marks:** code <font color="red">**[6 marks]**</font> — correct `generate_answers` (greedy, prompt-only decode), ROUGE-L + BERTScore scoring, and scoring all five runs into `results` / `preds_store`; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 9: Test Evaluation — IFT, LoRA, QLoRA
# ============================================================
import evaluate as hf_evaluate
rouge = hf_evaluate.load("rouge")

@torch.no_grad()
def generate_answers(model, frame, max_new_tokens=200, n=EVAL_N):
    """Greedy-decode an answer for each prompt. Returns (predictions, references)."""

    ## CODE HERE ##

    return preds, refs


def score(preds, refs):
    """ROUGE-L (word overlap) + BERTScore-F1 (meaning similarity)."""

    ## CODE HERE ##

    return {"rougeL": round(float(rl), 4), "bertscore_f1": round(float(Fb.mean()), 4)}


results, preds_store = {}, {}   # name -> metrics ; name -> predictions

In [ ]:
# ---- Full IFT ----
full_preds, test_refs = generate_answers(full_model, test_df)
results["full_ift"] = {**score(full_preds, test_refs),
                       "pct_trainable": round(pct_full, 2),
                       "peak_mem_gb": round(full_mem, 2),
                       "train_time_s": round(full_time, 1)}
preds_store["full_ift"] = full_preds
print("[full_ift]", results["full_ift"])

In [ ]:
# ---- LoRA and QLoRA, each at both ranks ----
for method, runs in [("lora", lora_runs), ("qlora", qlora_runs)]:
    for rank, run in runs.items():
        name = f"{method}_r{rank}"
        p, _ = generate_answers(run["model"], test_df)
        results[name] = {**score(p, test_refs),
                         "pct_trainable": round(run["pct"], 2),
                         "peak_mem_gb": round(run["mem"], 2),
                         "train_time_s": round(run["time"], 1)}
        preds_store[name] = p
        print(f"[{name}]", results[name])

### ✍️ Step 9 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- What ROUGE-L and BERTScore-F1 each measure, and why reporting both is more informative than either alone for free-form medical answers. <font color="red">**[2 marks]**</font>
- The scores across the five runs — which scores highest, and by how much over the others? <font color="red">**[1 marks]**</font>
- Since the references are machine-generated, why you should weigh *relative* differences between methods more than the *absolute* score values. <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

## <font color="blue"><b>Step 10: Plots — Train vs Validation Loss</b></font>  <font color="red">**[6 marks]**</font>

Visualise how each run trained. Draw one panel per run — full IFT, both LoRA ranks, and both QLoRA ranks — showing per-epoch **train** and **validation** loss, all sharing the same y-axis so loss levels are directly comparable across methods and ranks. The curves come from each run's `hist` (`train_loss` / `val_loss`).

**Marks:** code <font color="red">**[4 marks]**</font> — one train-vs-val panel per run, shared y-axis, correct labels/legend/titles; reflection <font color="red">**[2 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 10: Plots — Train vs Validation Loss (per epoch)
# ============================================================

## CODE HERE ##

### ✍️ Step 10 Reflection  <font color="red">**[2 marks]**</font>

**[Your Answer]** must cover:
- Which method converged fastest and to the lowest validation loss, and whether any run shows signs of **overfitting** (validation rising while training keeps falling). <font color="red">**[1 marks]**</font>
- Whether a lower *training* loss reliably predicted better *test* answers back in Step 9 — and what that tells you. <font color="red">**[1 marks]**</font>

*Cite the curves and the Step 9 numbers.*

## <font color="blue"><b>Step 11: Error Analysis</b></font>  <font color="red">**[8 marks]**</font>

Numbers alone don't tell you *how* a model is wrong. Take the best run by **ROUGE-L**, print a few of its generated answers next to the references, and back that up with a quantitative summary of the failure modes: predicted-vs-reference length, the length ratio (a quick proxy for rambling or truncation), and counts of empty / much-shorter / much-longer generations.

**Marks:** code <font color="red">**[4 marks]**</font> — select the best run by ROUGE-L, show sample generations vs references, and compute the length-ratio + failure-mode summary; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 11: Error Analysis
# ============================================================

## CODE HERE ##

In [ ]:
# ---- Quantitative summary of the errors ----
## CODE HERE ##

### ✍️ Step 11 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- One recurring way the model's answers differ from the references — length, specificity, hedging, or missing safety caveats — with a concrete example from the samples. <font color="red">**[2 marks]**</font>
- What the length-ratio and failure counts reveal about the model's behaviour (rambling, truncation, empty generations). <font color="red">**[1 marks]**</font>
- One concrete change you would try to reduce these errors (e.g. more epochs, higher rank, longer `MAX_LEN`, or different decoding settings). <font color="red">**[1 marks]**</font>

*Cite the samples and the summary numbers your run produced.*

## <font color="blue"><b>Step 12: Parameter Analysis</b></font>  <font color="red">**[6 marks]**</font>

Bring everything together into the cost–quality picture. Build one comparison table across all five runs — **quality** (ROUGE-L, BERTScore-F1) against **cost** (% trainable parameters, peak GPU memory, training time) — then draw the supporting charts: quality bars with peak memory annotated, and a trainable-% bar chart that makes the full-IFT-vs-adapter gap obvious.

**Marks:** code <font color="red">**[4 marks]**</font> — the comparison table over all runs and the quality + trainable-% charts; reflection <font color="red">**[2 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 12: Parameter Analysis
# ============================================================

# ---- Comparison table: quality vs cost across every run ----
## CODE HERE ##

In [ ]:
# ---- Quality chart with peak memory annotated ----
## CODE HERE ##

In [ ]:
# ---- Trainable-parameters comparison (the LoRA/QLoRA story) ----
## CODE HERE ##

### ✍️ Step 12 Reflection  <font color="red">**[2 marks]**</font>

**[Your Answer]** must cover:
- Which method gives the best **balance** of quality, trainable-%, memory, and training time for a free T4 — not simply the highest score — citing the comparison table. <font color="red">**[1 marks]**</font>
- How **rank** (8 vs 16) and **quantisation** (LoRA vs QLoRA) each traded answer quality against cost. <font color="red">**[1 marks]**</font>

*Cite the actual numbers from your comparison table.*

In [ ]:
# END OF THE NOTEBOOK --------------------------------